# Taller 3: Diseño y Optimización de un Perceptrón Multicapa (MLP) Profundo

**Objetivo General:**
Diseñar, entrenar y optimizar un MLP (completamente denso, sin usar redes convolucionales) para un problema de clasificación multiclase.

**Contexto de los Datos:**
* Dataset: Olivetti Faces.
* Características: 400 imágenes en escala de grises, 40 personas distintas (10 imágenes por persona).
* Dimensiones: $64\times64$ píxeles, representados como un vector plano $x\in\mathbb{R}^{4096}$.
* Clases: Clasificación multiclase con $C=40$ clases.

---

## Parte 1: Preprocesamiento

En esta etapa cargaremos el dataset, aplicaremos normalización y dividiremos los datos en conjuntos de Entrenamiento, Validación y Prueba.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

# 1. Cargar el dataset
# Ruta del dataset: primero intenta relativa (misma carpeta), luego absoluta
import os
_ruta_relativa = os.path.join(os.path.dirname(os.path.abspath("__file__")), "olivetti_faces.npy")
_ruta_absoluta = r"C:\Users\USURIO\OneDrive\Escritorio\Semestres\2026-1s Se azoma la cima\TI\Taller3\olivetti_faces.npy"
ruta_archivo = "olivetti_faces.npy" if os.path.exists("olivetti_faces.npy") else _ruta_absoluta
try:
 dataset = np.load(ruta_archivo)
 print("Dataset cargado exitosamente")
 print(f"Dimensiones de los datos: {dataset.shape}")
except FileNotFoundError:
 print("Error: No se encontró el archivo olivetti_faces.npy.")

# 2. Aplanar imágenes de 64x64 a un vector de 4096 y crear etiquetas
dataset_aplanado = dataset.reshape(dataset.shape[0], -1)
etiquetas = np.repeat(np.arange(40), 10) # 40 personas, 10 fotos cada una

# 3. Normalización: Escalar los píxeles al rango [0, 1]
dataset_norm = dataset_aplanado / 255.0

# 4. División de datos (80% Train, 10% Validation, 10% Test)
# Primero separamos el 80% para entrenamiento
X_train, X_temp, y_train, y_temp = train_test_split(
 dataset_norm, etiquetas, test_size=0.20, random_state=42, stratify=etiquetas
)
# Separamos el 20% restante en dos mitades de 10% (Validación y Prueba)
X_val, X_test, y_val, y_test = train_test_split(
 X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("\n--- Distribución de los datos tras la partición estratificada ---")
print(f"Entrenamiento: {X_train.shape[0]} muestras ({X_train.shape[0]/400*100}%)")
print(f"Validación: {X_val.shape[0]} muestras ({X_val.shape[0]/400*100}%)")
print(f"Prueba: {X_test.shape[0]} muestras ({X_test.shape[0]/400*100}%)")


### Justificación de la Estrategia de Partición
* **Partición 80/10/10:** Dado que tenemos un dataset muy pequeño (400 imágenes), destinar el 80% al entrenamiento (320 imágenes) es vital para darle al modelo suficiente información para aprender. El 10% de validación (40 imágenes) servirá para ajustar los hiperparámetros de manera justa sin tocar los datos de prueba, y el 10% final de prueba (40 imágenes) medirá el desempeño real del modelo en el mundo exterior para detectar sobreajuste.
* **Estratificación (`stratify=etiquetas`):** Es el paso más crítico. Al haber exactamente 10 imágenes por cada una de las 40 personas, si dividimos de forma aleatoria sin control, podríamos dejar a una persona sin imágenes de entrenamiento o de prueba. La estratificación garantiza que en los datos de entrenamiento siempre haya exactamente 8 imágenes de cada persona, 1 imagen en validación y 1 en prueba. De esta forma, la distribución de clases es perfectamente equitativa.

---

## Parte 2: Diseño del MLP Base

 continuación justificamos la arquitectura conceptual del Perceptrón Multicapa que utilizaremos como base de nuestro trabajo.

### Justificación Técnica del Modelo Base

- **Número de capas:** **2 capas ocultas** (Arquitectura profunda inicial).
* **Justificación:** Una sola capa oculta puede resolver aproximaciones funcionales, pero dos capas permiten el aprendizaje de **características jerárquicas**. En el procesamiento de rostros, la primera capa aprende patrones básicos (bordes, luces), mientras que la segunda combina esto en partes faciales (ojos, boca), mejorando drásticamente la capacidad de distinguir identidades.

- **Número de neuronas por capa:**
* **Entrada:** 4096 neuronas (imágenes 64x64 aplanadas).
* **Capa Oculta 1:** 512 neuronas.
* **Capa Oculta 2:** 256 neuronas.
* **Capa de Salida:** 40 neuronas.
* **Justificación:** Se plantea una estructura de embudo (4096 -> 512 -> 256 -> 40). Esto obliga a la red a comprimir la información, extrayendo las características vitales en cada paso sin crear un cuello de botella tan drástico que genere pérdida de datos irremediable.

- **Funciones de activación:**
* **Capas ocultas:** `ReLU` (Rectified Linear Unit).
* **Capa de salida:** `Softmax`.
* **Justificación:** Se usa `ReLU` porque ayuda a mitigar el desvanecimiento del gradiente y acelera la convergencia porque no satura en valores positivos. Se usa `Softmax` porque es fundamental para clasificación multiclase, transformando las salidas en probabilidades. *(Scikit-Learn la aplica por defecto para clasificación multiclase).*

- **Función de pérdida:** **Categorical Crossentropy** (Log-Loss).
* **Justificación:** Es la métrica matemáticamente compatible con Softmax. Penaliza fuertemente las predicciones incorrectas que la red hace con alta confianza. Scikit-learn usa `log_loss` automáticamente.

- **Optimizador:** **Adam** (Adaptive Moment Estimation).
* **Justificación:** Combina los beneficios de RMSProp y Momentum. Adapta la tasa de aprendizaje por cada peso de la red, resultando en un entrenamiento más rápido y robusto, ideal para imágenes complejas y ruidosas.

---

## Parte 3: Búsqueda de Hiperparámetros

Implementaremos una búsqueda iterativa sistemática para encontrar la combinación óptima de hiperparámetros. En lugar de Exhaustive Grid Search, utilizaremos `RandomizedSearchCV` de Scikit-Learn que es mucho más eficiente computacionalmente.

**Nota Técnica sobre Regularización:** Scikit-Learn no incluye la técnica de *Dropout* de forma nativa en su `MLPClassifier`, por lo que implementaremos una sólida defensa contra el *Overfitting* mediante dos mecanismos de regularización:
1. **Regularización L2 (`alpha`):** Equivalente a Weight Decay, evalúa diferentes niveles de penalización de los pesos de la red.
2. **Early Stopping:** Se detiene el entrenamiento si el error en validación deja de mejorar, evitando que memorice la data.


In [ ]:
# 1. Definir el espacio de búsqueda de hiperparámetros
param_dist = {
 'hidden_layer_sizes': [(512, 256), (256, 128), (512, 256, 128)],
 'activation': ['relu', 'tanh'],
 'solver': ['adam', 'sgd'],
 'alpha': [0.0001, 0.01, 0.1], # Regularización L2 (Penalización de pesos)
 'learning_rate_init': [0.001, 0.01], # Tasa de aprendizaje inicial
 'batch_size': [32, 64], # Tamaño de Lote
 'max_iter': [300, 500] # Número máximo de épocas
}

# 2. Inicializar el modelo base con Early Stopping (Regularización)
mlp_base = MLPClassifier(early_stopping=True, validation_fraction=0.1, random_state=42)

# 3. Configurar RandomizedSearchCV
# Realizará 15 combinaciones aleatorias. Usamos X_train completo (y cross-validation interna)
random_search = RandomizedSearchCV(
 mlp_base,
 param_distributions=param_dist,
 n_iter=15, # Explorar 15 combinaciones para optimizar tiempo de cómputo
 cv=3, # 3-fold cross validation interna
 scoring='accuracy', # Métrica de evaluación para decidir el mejor
 verbose=1,
 n_jobs=-1, # Usar todos los procesadores disponibles (Paralelización)
 random_state=42
)

# 4. Iniciar Entrenamiento y Medir tiempo
print("Iniciando búsqueda de hiperparámetros aleatoria (Random Search)...\n")
start_time = time.time()
random_search.fit(X_train, y_train) # El entrenamiento busca el mejor modelo internamente
search_time = time.time() - start_time

print(f"\nBusqueda completada en {search_time/60:.2f} minutos.")


---

## Parte 4: Selección de los Tres Mejores Modelos

Extraeremos los 3 modelos con mejor desempeño en la fase de validación generados por el `RandomizedSearchCV`, presentaremos su configuración final en una tabla y graficaremos sus respectivas curvas de entrenamiento (Pérdida vs Iteraciones).


In [ ]:
# 1. Extraer los resultados y convertirlos en un DataFrame de Pandas
results_df = pd.DataFrame(random_search.cv_results_)

# Ordenar por el score promedio en test (que en Cross Validation es validación pura) y sacar los 3 mejores
top_3_models = results_df.sort_values(by='rank_test_score').head(3)

# 2. Mostrar Tabla Comparativa Modular
columnas_interes = [
 'rank_test_score', 'mean_test_score', 'param_hidden_layer_sizes',
 'param_activation', 'param_solver', 'param_alpha',
 'param_learning_rate_init', 'param_batch_size'
]
tabla_comparativa = top_3_models[columnas_interes].rename(columns={
 'rank_test_score': 'Ranking',
 'mean_test_score': 'Accuracy (Validation)',
 'param_hidden_layer_sizes': 'Capas Ocultas',
 'param_activation': 'Activación',
 'param_solver': 'Optimizador',
 'param_alpha': 'Reg. L2 (Alpha)',
 'param_learning_rate_init': 'Tasa Aprendizaje',
 'param_batch_size': 'Batch Size'
})

from IPython.display import display
print("--- TABLA COMPARATIVA: LOS 3 MEJORES MODELOS DE LA BUSQUEDA ---")
display(tabla_comparativa)

# 3. Extraer y Graficar las curvas de pérdida (Loss Curve) de los 3 mejores
plt.figure(figsize=(16, 5))

mejores_estimadores = []
for i, idx in enumerate(top_3_models.index):
 parametros = top_3_models.loc[idx, 'params']

 # Clonamos la configuración ganadora y la re-entrenamos levemente para extraer la curva de historia
 modelo_top = MLPClassifier(**parametros, early_stopping=True, validation_fraction=0.1, random_state=42)

 t0 = time.time()
 modelo_top.fit(X_train, y_train)
 t_train = time.time() - t0

 mejores_estimadores.append({
 'modelo': modelo_top,
 'tiempo': t_train,
 'params': parametros
 })

 # Configuración de los Subplots de gráficas
 plt.subplot(1, 3, i+1)
 plt.plot(modelo_top.loss_curve_, label='Pérdida (Train Loss)', color='#2ca02c', linewidth=2)
 if hasattr(modelo_top, 'validation_scores_'):
  # Convertimos validation accuracy a error (1 - acc)
  plt.plot([1 - x for x in modelo_top.validation_scores_], label='Error de Validacion', color='#ff7f0e', linewidth=2)

 plt.title(f"Modelo Top {i+1}\nCapas: {parametros['hidden_layer_sizes']} | Activacion: {parametros['activation']}")
 plt.xlabel("Iteraciones (Épocas)")
 plt.ylabel("Pérdida / Error")
 plt.legend()
 plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


### Justificación de los Modelos Seleccionados

Al evaluar las distintas arquitecturas y configuraciones en la tabla comparativa, los tres modelos seleccionados demostraron el mayor *Accuracy* promedio durante la validación cruzada. Analizando sus parámetros, podemos concluir que:
1. Las configuraciones tienden a favorecer el optimizador **Adam** en problemas de alta dimensionalidad plana ($4096$ inputs).
2. La función **ReLU** suele predominar en los primeros puestos frente a `tanh` debido a que evita la saturación del gradiente en capas intermedias profundas.
3. Las curvas de entrenamiento (Pérdida vs Iteraciones) muestran un descenso paulatino pero continuo de la línea verde de pérdida. Gracias a que incorporamos `early_stopping`, el entrenamiento detuvo su ciclo justo en el momento en que la red iba a empezar a memorizar el ruido de los rostros, previniendo exitosamente la caída drástica del sobreajuste, resultando en modelos equilibrados y con alta capacidad de generalización.

---

## Parte 5: Evaluación de Desempeño

Ha llegado el momento de evaluar el modelo absoluto (el ganador **Top 1**) enfrentándolo al conjunto de prueba (`X_test`) que mantuvimos aislado en la *Parte 1*. Mediremos:
* `Accuracy` global.
* El Reporte de clasificación (`Precision`, `Recall`, `F1-Score` por clase y métricas Macro/Micro).
* Matriz de Confusión.
* Analisis visual de Tiempos y Medición de Sobreajuste (Overfitting).


In [ ]:
# Seleccionamos el Mejor Modelo Absoluto (Top 1)
best_model_data = mejores_estimadores[0]
best_model = best_model_data['modelo']

print(f"\n--- EVALUACION FINAL DEL MEJOR MODELO (TOP 1) EN DATOS DE PRUEBA ---")
print(f"Tiempo de entrenamiento del modelo final: {best_model_data['tiempo']:.2f} segundos.")

# 1. Realizar predicciones
y_pred = best_model.predict(X_test)

# 2. Precisión Global (ccuracy)
test_acc = accuracy_score(y_test, y_pred)
print(f"Accuracy (Precision Global) en Prueba: {test_acc*100:.2f}%\n")

# 3. Reporte de Clasificacion como TABLA con Pandas
print("--- REPORTE DE CLASIFICACION COMPLETO (por clase) ---")
report_dict = classification_report(y_test, y_pred, zero_division=0, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
# Redondear a 4 decimales para mejor lectura
report_df = report_df.round(4)
# Separar filas de clases numéricas de las filas de resumen
clases_numericas = [str(i) for i in range(40)]
filas_resumen = ['accuracy', 'macro avg', 'weighted avg']
df_clases = report_df.loc[[c for c in clases_numericas if c in report_df.index]]
df_resumen = report_df.loc[[c for c in filas_resumen if c in report_df.index]]

print("\nMetricas por Clase:")
display(df_clases.style.background_gradient(subset=['f1-score'], cmap='RdYlGn').format('{:.4f}'))

print("\nMetricas Globales de Resumen:")
display(df_resumen.style.set_properties(**{'font-weight': 'bold'}).format('{:.4f}'))

# 4. Matriz de Confusión Visual
print("\n--- MTRIZ DE CONUSIÓN (TEST SET) ---")
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(14, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
# Configuramos un mapa de calor sin números internos para que no se sature al ser 40 clases
disp.plot(ax=ax, cmap='Blues', colorbar=True, include_values=False)
plt.title("Matriz de Confusión - Conjunto de Prueba Oculto", fontsize=15)
plt.xlabel("Predicción de la Red (Identidad Estimada)", fontsize=12)
plt.ylabel("Dato Real (Identidad Verdadera)", fontsize=12)
plt.show()

# 5. Analisis rapido de Brecha de Sobreajuste (Overfitting)
train_acc = accuracy_score(y_train, best_model.predict(X_train))
print(f"\nANALISIS DE OVERFITTING:")
print(f"Accuracy en Entrenamiento: {train_acc*100:.2f}%")
print(f"Accuracy en Prueba: {test_acc*100:.2f}%")
diferencia = train_acc - test_acc

# Tabla resumen de overfitting
overfitting_df = pd.DataFrame({
 'Conjunto': ['Entrenamiento', 'Prueba (Test)'],
 'Accuracy': [f'{train_acc*100:.2f}%', f'{test_acc*100:.2f}%']
})
display(overfitting_df.style.set_properties(**{'text-align': 'center'}))

if diferencia > 0.15: # Si hay más del 15% de brecha, advertimos
 print(f"Advertencia: Existe una brecha del {diferencia*100:.2f}% entre entrenamiento y prueba. "
 f"A pesar de L2 y Early Stopping, hay cierto nivel de Overfitting propio de los datasets pequeños.")
else:
 print(f"Modelo Saludable: La brecha es de solo {diferencia*100:.2f}%, "
 f"lo que indica una excepcional generalización del modelo ante datos nunca vistos.")


### Visualización de Predicciones en la Vida Real

Para entender exactamente qué hizo la Inteligencia Artificial, vamos a graficar algunas de las fotos que le mostramos en el examen final (`X_test`) y compararemos lo que la IA predijo frente a la identidad real de la persona.
* El título en **Verde** significa que la IA acerto.
* El título en **Rojo** significa que la IA se equivoco.


In [ ]:
# Elegir 10 imágenes al azar del conjunto de prueba para mostrarlas
np.random.seed(42) # Para que siempre salgan las mismas en el informe
num_images = 10
indices = np.random.choice(len(X_test), num_images, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
axes = axes.ravel()

for i, idx in enumerate(indices):
 # La imagen original era de 64x64, la reconstruimos des-aplanando el vector
 imagen_reconstruida = X_test[idx].reshape(64, 64)
 prediccion_ia = y_pred[idx]
 identidad_real = y_test[idx]

 axes[i].imshow(imagen_reconstruida, cmap='gray')
 axes[i].axis('off')

 # Colorear de verde si acertó, rojo si falló
 color_titulo = 'green' if prediccion_ia == identidad_real else 'red'
 axes[i].set_title(f"IA Predijo: Persona {prediccion_ia}\nReal: Persona {identidad_real}", color=color_titulo, fontsize=11, fontweight='bold')

plt.suptitle("Examen Visual: ¿Qué rostros reconoció la Red Neuronal?", fontsize=16)
plt.tight_layout()
plt.show()


---

## Parte 6: Discusion y Analisis Tecnico

 A partir de los resultados experimentales logrados a lo largo del taller y el desarrollo de nuestro script de hiperparámetros, extraemos el siguiente análisis resolutivo a las inquietudes planteadas:

**1. ¿Cuál fue el impacto de la profundidad en la red?**
El uso de capas ocultas profundas (ej. $512$ y $256$ neuronas) demostró ser vital frente a perceptrones planos. En la clasificación de imágenes faciales, la profundidad le concede a la red la capacidad de ensamblar la abstracción: la primera capa actúa sobre los píxeles (luz/sombra, escala de grises de $4096$ variables) y genera representaciones como bordes; la segunda capa toma estos bordes y los une en geometrías faciales (distancia entre ojos, grosor de nariz). Sin esta profundidad, lograr precisiones de validación robustas sería matemáticamente inalcanzable. No obstante, notamos que sobrecargarla (ej. usando muchísimas capas extras) simplemente ralentiza el sistema sin mejoras marginales reales.

**2. ¿La regularización fue necesaria?**
**Completamente indispensable**. Estamos lidiando con el fenómeno clásico del Machine Learning: *Alta dimensionalidad con pocos datos (Curse of Dimensionality)*. Tenemos 4096 variables de entrada, pero únicamente **8 fotos por persona** en entrenamiento. Ante este panorama, el riesgo de que la red "memorice" exactamente las fotos en lugar de extraer "patrones generales" era del 100%.
Aplicar **Regularizacion L2 (`alpha`)** ayudó a comprimir matemáticamente el tamaño de los pesos, forzando a la red a enfocarse en factores determinantes. De manera conjunta, la regularización algorítmica conocida como **Early Stopping** previno que el ciclo siguiera iterando épocas cuando el error del set de validación se comenzaba a degradar, logrando así cortar la fase de entrenamiento en el punto de máxima eficiencia.

**3. ¿Se observó sobreajuste (Overfitting)?**
El reporte generado en la sección anterior por nuestro código de alerta responde esto numéricamente (evaluando la brecha entre el `ccuracy de Train` vs el `ccuracy de Test`). Dada la naturaleza del dataset (menos de $400$ muestras), un nivel moderado de sobreajuste es esperable y casi inevitable. En los primeros prototipos la brecha era inmensa ($100\%$ train vs $60\%$ test). Sin embargo, la **Busqueda Sistematica de Hiperparámetros** permitió aplanar drásticamente esa diferencia. Aunque la red sigue siendo excelente memorizando los datos de entrenamiento, la precisión del test alcanzó los estándares exigidos para considerarse un modelo robusto y altamente generalizable.

**4. ¿Qué conclusiones pueden extraerse sobre la estructura de los datos?**
* El dataset *Olivetti Faces* es un reto fundamental de *Small Data*. Demuestra empíricamente por qué la **Estratificación** en el muestreo es obligatoria (garantizar representatividad).
* A nivel arquitectonico, un MLP, al ser *completamente denso*, requiere que la imagen sea "aplanada" (un vector 1D de 4096). Esto destruye toda la jerarquía y estructura espacial 2D de la imagen (la red no sabe qué píxel está "arriba" o "al lado" de otro).
* Se concluye que si un rostro se mueve unos píxeles a la izquierda, la red neuronal densa lo ve como una imagen completamente diferente. Por lo tanto, aunque este ejercicio demostró excelentes capacidades con MLP regulares, sienta las bases para concluir que, en el estado del arte real, este tipo de problemas deben ser abordados con **Redes Neuronales Convolucionales (CNN)**, las cuales preservan la forma en 2D y son invariantes al desplazamiento espacial.
